# Clasificación — Predicción de clase de temperatura
- **13 clases**: una por temperatura exacta
- **3 clases**: frío (<30°C), templado (30–55°C), caliente (>55°C)

Modelos: Random Forest, Gradient Boosting, SVM, MLP

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

CSV_PATH  = Path("dataset_features_temperatura.csv")
FEAT_JSON = Path("features_seleccionadas.json")

In [ ]:
df = pd.read_csv(CSV_PATH)
META_COLS = {"sample_uid","sample_id","tx_path","rx_path",
    "tx_path_rel_to_bbdd_parent","rx_path_rel_to_bbdd_parent",
    "split","temperature_folder","N_subcarriers","M_symbols","num_valid","valid_ratio"}
CONSTANT_COLS = {"phase_jump_m_count","phase_jump_m_ratio","phase_jump_k_count","phase_jump_k_ratio"}

if FEAT_JSON.exists():
    with open(FEAT_JSON) as f: FEAT_COLS = json.load(f)
    print(f"Features desde JSON: {len(FEAT_COLS)}")
else:
    FEAT_COLS = [c for c in df.columns if c not in META_COLS and c != "temperature" and c not in CONSTANT_COLS]
    print(f"Usando todas las features: {len(FEAT_COLS)}")

df["label_13"] = df["temperature"].astype(int).astype(str) + "C"
df["label_3"]  = df["temperature"].apply(lambda t: "frio" if t<30 else ("templado" if t<=55 else "caliente"))

def get_split(df, split, label):
    d = df[df["split"]==split]
    return d[FEAT_COLS].values, d[label].values

X_tr13,y_tr13 = get_split(df,"train","label_13"); X_v13,y_v13 = get_split(df,"val","label_13"); X_te13,y_te13 = get_split(df,"test","label_13")
X_tr3, y_tr3  = get_split(df,"train","label_3");  X_v3, y_v3  = get_split(df,"val","label_3");  X_te3, y_te3  = get_split(df,"test","label_3")
classes_13 = sorted(df["label_13"].unique()); classes_3 = ["frio","templado","caliente"]
print("Clases 13:", classes_13)
print("Distribucion 3 clases (train):"); print(df[df["split"]=="train"]["label_3"].value_counts().to_string())

In [ ]:
def make_models():
    return {
        "Random Forest": Pipeline([("imp",SimpleImputer(strategy="mean")),("model",RandomForestClassifier(n_estimators=200,min_samples_leaf=2,n_jobs=-1,random_state=42))]),
        "Gradient Boosting": Pipeline([("imp",SimpleImputer(strategy="mean")),("model",GradientBoostingClassifier(n_estimators=200,learning_rate=0.05,max_depth=5,subsample=0.8,random_state=42))]),
        "SVM (RBF)": Pipeline([("imp",SimpleImputer(strategy="mean")),("scaler",StandardScaler()),("model",SVC(kernel="rbf",C=10,gamma="scale",probability=True,random_state=42))]),
        "MLP": Pipeline([("imp",SimpleImputer(strategy="mean")),("scaler",StandardScaler()),("model",MLPClassifier(hidden_layer_sizes=(128,64,32),activation="relu",max_iter=300,early_stopping=True,random_state=42))]),
    }

In [ ]:
print("=" * 50); print("13 CLASES"); print("=" * 50)
results_13 = {}; models_13 = make_models()
for name, model in models_13.items():
    print(f"{name}...", end=" ", flush=True)
    model.fit(X_tr13, y_tr13)
    preds = model.predict(X_te13)
    acc = accuracy_score(y_te13, preds)
    results_13[name] = {"acc": acc, "preds": preds}
    print(f"Acc={acc*100:.2f}%")
best_13 = max(results_13, key=lambda n: results_13[n]["acc"])
acc_best = results_13[best_13]["acc"]
print(f"Mejor: {best_13} -> {acc_best*100:.2f}%")

In [ ]:
print("=" * 50); print("3 CLASES"); print("=" * 50)
results_3 = {}; models_3 = make_models()
for name, model in models_3.items():
    print(f"{name}...", end=" ", flush=True)
    model.fit(X_tr3, y_tr3)
    preds = model.predict(X_te3)
    acc = accuracy_score(y_te3, preds)
    results_3[name] = {"acc": acc, "preds": preds}
    print(f"Acc={acc*100:.2f}%")
best_3 = max(results_3, key=lambda n: results_3[n]["acc"])
acc_best3 = results_3[best_3]["acc"]
print(f"Mejor: {best_3} -> {acc_best3*100:.2f}%")

In [ ]:
model_names = list(results_13.keys())
acc_13 = [results_13[n]["acc"]*100 for n in model_names]
acc_3  = [results_3[n]["acc"]*100  for n in model_names]
x = np.arange(len(model_names)); w = 0.35
fig, ax = plt.subplots(figsize=(13, 6))
b1 = ax.bar(x-w/2, acc_13, w, label="13 clases", color="#2E75B6", alpha=0.85)
b2 = ax.bar(x+w/2, acc_3,  w, label="3 clases",  color="#70AD47", alpha=0.85)
for bar,val in zip(list(b1)+list(b2), acc_13+acc_3):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5, f"{val:.1f}%", ha="center", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(model_names,fontsize=10)
ax.set_ylabel("Accuracy [%]",fontsize=11); ax.set_ylim(0,115)
ax.set_title("Comparativa clasificadores — 13 vs 3 clases",fontsize=12,fontweight="bold")
ax.legend(fontsize=10); ax.grid(axis="y",alpha=0.3)
plt.tight_layout(); plt.savefig("clasificacion_accuracy.png",dpi=150,bbox_inches="tight"); plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(22, 18)); axes = axes.flatten()
for i, (name, r) in enumerate(results_13.items()):
    cm = confusion_matrix(y_te13, r["preds"], labels=classes_13)
    ConfusionMatrixDisplay(cm, display_labels=classes_13).plot(ax=axes[i], colorbar=True, cmap="Blues", values_format="d")
    acc = r["acc"]
    axes[i].set_title(f"{name}  Acc={acc*100:.2f}%", fontsize=10, fontweight="bold")
    axes[i].tick_params(axis="x", rotation=45, labelsize=8)
plt.suptitle("Matrices de confusion — 13 clases (test)", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.savefig("clasificacion_confusion_13.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12)); axes = axes.flatten()
for i, (name, r) in enumerate(results_3.items()):
    cm = confusion_matrix(y_te3, r["preds"], labels=classes_3)
    ConfusionMatrixDisplay(cm, display_labels=classes_3).plot(ax=axes[i], colorbar=True, cmap="Greens", values_format="d")
    acc = r["acc"]
    axes[i].set_title(f"{name}  Acc={acc*100:.2f}%", fontsize=10, fontweight="bold")
plt.suptitle("Matrices de confusion — 3 clases (test)", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.savefig("clasificacion_confusion_3.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
print(f"REPORT {best_13} — 13 clases")
print(classification_report(y_te13, results_13[best_13]["preds"], target_names=classes_13))
print(f"REPORT {best_3} — 3 clases")
print(classification_report(y_te3, results_3[best_3]["preds"], target_names=classes_3))

In [ ]:
importances = pd.Series(models_13["Random Forest"].named_steps["model"].feature_importances_, index=FEAT_COLS).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10,12))
top20 = importances.head(20)
ax.barh(top20.index[::-1], top20.values[::-1], color="#2E75B6", alpha=0.85)
ax.set_xlabel("Importancia (Gini)",fontsize=10)
ax.set_title("Top 20 features — Random Forest (13 clases)",fontsize=11,fontweight="bold")
ax.grid(axis="x",alpha=0.3)
plt.tight_layout(); plt.savefig("clasificacion_importancia.png",dpi=150,bbox_inches="tight"); plt.show()